In [1]:
# import the required modules
import re
from bs4 import BeautifulSoup

# load the sample data
filename = "25-dianchi-dr-wyndham-vale-vic-3024.html"
with open("../data/samples/rent_dotcom_html/"+filename, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

In [2]:
import json, re
from bs4 import BeautifulSoup

DEC = json.JSONDecoder()
SIGNATURE = {"id", "bedrooms", "bathrooms", "suburb", "postcode",
             "weekly_rent_number", "latitude", "longitude", "property_type"}


def flight_payload(soup):
    chunks = []
    for tag in soup.find_all("script"):
        txt = tag.string or ""
        if "__next_f.push" not in txt:
            continue
        m = re.search(r"__next_f\.push\((\[.*\])\)", txt, re.DOTALL)
        if not m:
            continue
        try:
            arr = json.loads(m.group(1))
        except json.JSONDecodeError:
            continue
        if len(arr) > 1 and isinstance(arr[1], str):
            chunks.append(arr[1])
    return "".join(chunks)


def get_listing(payload):
    for m in re.finditer(r'"property"\s*:', payload):
        try:
            obj, _ = DEC.raw_decode(payload, payload.index("{", m.end() - 1))
        except (ValueError, json.JSONDecodeError):
            continue
        if isinstance(obj, dict) and len(SIGNATURE & obj.keys()) >= 4:
            return obj
    raise ValueError(f"no listing found in {len(payload):,} chars of payload")


payload = flight_payload(soup)
listing = get_listing(payload)

**Note: some features with high prediction potential have been dropped since they are derived from elsewhere and would potentially dominate the model and our features.**

In [3]:

keys = [
    "id", "url", "latitude", "longitude", "street_address", "suburb", "postcode",
    "state", "activated_at", "weekly_rent_number", "property_type", "bedrooms",
    "bathrooms", "ensuites", "car_spaces", "furnished", "partially_furnished",
    "pets_allowed", "long_term_lease", "is_build_to_rent", "features",
]

record = {k: listing.get(k) for k in keys}
record["n_photos"] = len(listing.get("gallery") or [])
record["external_links"] = listing.get("external_links")
record["walkability_score"] = listing.get("walkability_score")
record["top_4_features"] = listing.get("top_4_features")

for k, v in record.items():
    print(k, v)

id 6480562
url https://www.rent.com.au/property/25-dianchi-dr-wyndham-vale-vic-3024-6480562
latitude -37.872968
longitude 144.619508
street_address 25 Dianchi Drive, 
suburb wyndham vale
postcode 3024
state vic
activated_at 2026-09-04T16:33:24.000+08:00
weekly_rent_number 530
property_type House
bedrooms 4
bathrooms 2
ensuites 0
car_spaces 2
furnished False
partially_furnished False
pets_allowed None
long_term_lease False
is_build_to_rent False
features {'garage': ['remote garage', 'secure parking'], 'heating': ['ducted heating'], 'kitchen': ['dishwasher'], 'flooring': ['floorboards flooring'], 'security': ['fully fenced'], 'airconditioning': ['ducted', 'evaporative']}
n_photos 20
external_links [{'url': 'https://youtu.be/DkqqS0xAkjY', 'category': 'video', 'youtube_id': 'DkqqS0xAkjY'}]
walkability_score {'id': 4705589, 'walk_score': 51, 'walk_category': 'somewhat_walkable', 'transit_score': 39, 'transit_category': 'some_transit'}
top_4_features [{'name': 'nature', 'description': 'Parks